# 模块 2 节点壳：Tracker ROS 消息转换

对应 `tracker_node.py`，仅演示 `CentroidArray -> list[np.ndarray] -> TrackState -> Track -> TrackArray`、时间戳语义和话题接口。

> **运行前提**：使用 **ROS Noetic (Py3.8)** 内核，并已执行 `source /opt/ros/noetic/setup.bash`、`source catkin_ws/devel/setup.bash`及 `catkin_make`。本 notebook 不初始化 ROS 节点、不启动 master，不调用 `spin()`。

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'catkin_ws').is_dir():
    root = root.parent
scripts = root / 'catkin_ws' / 'src' / 'uav_tracking' / 'scripts'
assert scripts.is_dir(), '未找到 ROS 工作区'
sys.path.insert(0, str(scripts))

import rospy
from geometry_msgs.msg import PointStamped
from uav_tracking.msg import CentroidArray
import tracker_node as node
from tracker import KalmanTracker, track_state_to_track

print('input :', node.TOPIC_CENTROIDS, 'uav_tracking/CentroidArray')
print('output:', node.TOPIC_TRACKS, 'uav_tracking/TrackArray')
print('mode  : event-driven; no periodic timer')

## 1. `CentroidArray` 拆包及量测时间戳

消息时间戳只记录量测更新时间；预测 `dt` 由节点回调中相邻 ROS 接收时间差计算。

In [ ]:
centroids = CentroidArray()
for xyz in ((1.0, 2.0, 3.0), (4.0, -1.0, 2.5)):
    point = PointStamped()
    point.header.stamp = rospy.Time.from_sec(12.5)
    point.point.x, point.point.y, point.point.z = xyz
    centroids.centroids.append(point)
measurements = node.centroids_from_msg(centroids)
print('measurements     :', measurements)
print('measurement stamp:', node.stamp_to_seconds(centroids))
print('empty input      :', node.centroids_from_msg(CentroidArray()))
assert node.stamp_to_seconds(centroids) == 12.5

## 2. 三层转换到 `TrackArray`

算法返回 `TrackState`，纯转换得到 `Track`，节点壳再装包为 ROS `Track.msg`。

In [ ]:
tracker = KalmanTracker()
states = tracker.step([measurements[0]], dt=1.0 / 30.0, timestamp=12.5)
tracks = [track_state_to_track(state) for state in states]
message = node.tracks_to_msg(tracks)
item = message.tracks[0]
print('TrackState:', states[0])
print('Track     :', tracks[0])
print('ROS Track : id=%d position=%s age=%d covariance_len=%d' %
      (item.id, list(item.position), item.age, len(item.covariance)))
print('empty publish length:', len(node.tracks_to_msg([]).tracks))
assert item.id == tracks[0].id and len(item.covariance) == 36